# Repaso integrador: clase 8
### BCD5105 Modelado matemático · III cuatrimestre 2026 · Jueves 29 de octubre

Hoy repasamos los temas 1 a 7 con **un solo caso real**: la electricidad de Costa Rica, mes a mes, desde enero de 2019. Los datos no vienen en el notebook: se descargan con código desde la fuente oficial, igual que en la prueba parcial.

El recorrido es el mismo que pide la prueba:

**descargar → explorar → modelar → validar → elegir → pronosticar**

En el camino aparecen los demás temas del curso, aplicados a los mismos datos.

**Regla de oro:** las celdas se corren en orden y cada una produce una salida visible (un gráfico o un `print`).

## Mapa del repaso

| Sección | Tema del curso | Pregunta sobre la electricidad de Costa Rica |
|---|---|---|
| 1 | Fundamentos | ¿Qué datos tenemos y qué supuestos hacemos? |
| 2 | Exploración | ¿Hay tendencia y estacionalidad? ¿Qué pasó en 2024? |
| 3 | Ecuaciones diferenciales (tema 2) | ¿A qué ritmo crece la demanda? |
| 4 | Cadenas de Markov (tema 3) | ¿Con qué frecuencia el país necesita plantas térmicas? |
| 5 | Regresión y regularización (tema 5) | ¿Cuánto explica la tendencia y el mes del año? |
| 6 | Series clásicas y modernas (temas 6 y 7) | ¿Qué modelo pronostica mejor la demanda? |
| 7 | Monte Carlo (tema 4) | ¿Qué tan probable es superar cierto consumo el próximo año? |
| 8 | Pronóstico final | ¿Cuánta electricidad pedirá el país de julio de 2026 a junio de 2027? |

## 0. Preparar el entorno

Prophet y Chronos-2 no vienen en Colab. La instalación tarda un par de minutos: córranla primero.

In [ ]:
!pip install -q prophet chronos-forecasting

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings, logging
warnings.filterwarnings("ignore")
logging.getLogger("cmdstanpy").setLevel(logging.ERROR)

NARANJA, OSCURO, GRIS = "#FF871E", "#262626", "#8A8A8A"
plt.rcParams.update({"figure.figsize": (10, 4), "axes.spines.top": False, "axes.spines.right": False})
rng = np.random.default_rng(2026)   # semilla del curso
print("Entorno listo.")

## 1. Los datos: descargarlos de la fuente

**Fuente:** Ember, *Monthly Electricity Data* (licencia CC BY 4.0). Ember recopila datos oficiales de generación y demanda eléctrica de unos 90 países y publica un CSV abierto que se actualiza cada mes. Para Costa Rica hay datos mensuales desde enero de 2019.

El archivo trae todos los países en formato largo (una fila por país, mes y variable). Lo filtramos y lo pasamos a formato ancho: una fila por mes y una columna por variable. Convertimos de TWh a GWh para leer mejor (1 TWh = 1000 GWh).

In [ ]:
URL = ("https://storage.googleapis.com/emb-prod-bkt-publicdata/"
       "public-downloads/monthly_full_release_long_format.csv")
cols = ["Area", "Date", "Category", "Variable", "Unit", "Value"]
crudo = pd.read_csv(URL, usecols=cols, parse_dates=["Date"])   # unos 70 MB, tarda unos segundos
print("Filas del archivo completo:", len(crudo))

cr = crudo[(crudo["Area"] == "Costa Rica") & (crudo["Unit"] == "TWh")]
cr = (cr.pivot_table(index="Date", columns="Variable", values="Value") * 1000).asfreq("MS")
cr.index.name = "mes"
cr = cr[["Demand", "Hydro", "Wind", "Other Renewables", "Bioenergy", "Solar", "Fossil"]]
cr.columns = ["demanda", "hidro", "eolica", "geotermica_otras", "bio", "solar", "fosil"]
cr = cr[:"2026-06"]   # corte fijo: así todos obtienen los mismos números aunque Ember agregue meses

cr.to_csv("ember_costa_rica_mensual.csv")   # copia local, por si la fuente cambia
print("Meses:", len(cr), "| desde", cr.index.min().date(), "hasta", cr.index.max().date())
cr.tail()

**Supuestos y cuidados (tema 1).** Antes de modelar conviene anotar lo que sabemos de los datos:

- Son **datos oficiales agregados**, redondeados a 10 GWh (Ember publica en TWh con dos decimales). Una diferencia de 10 GWh puede ser solo redondeo.
- "Otras renovables" en Costa Rica es, sobre todo, **geotermia**.
- Cada mes Ember agrega datos y puede **revisar** cifras recientes: por eso cortamos en junio de 2026 y guardamos una copia local.
- Son 90 meses: suficiente para ver estacionalidad, poco para modelos que piden miles de datos.

## 2. Exploración: ¿qué cuentan los datos?

In [ ]:
fig, ax = plt.subplots()
ax.plot(cr.index, cr["demanda"], color=NARANJA, marker="o", ms=3)
ax.set_title("Demanda eléctrica mensual de Costa Rica (GWh)")
ax.set_ylabel("GWh"); ax.grid(alpha=.3)
plt.show()

anual = cr["demanda"].groupby(cr.index.year).sum()
print("Demanda anual (GWh), solo años completos:")
print(anual[anual.index <= 2025].round(0))

La demanda crece año con año (con un bache en 2020 por la pandemia) y cada año tiene picos y valles parecidos. Veamos de dónde sale la electricidad.

In [ ]:
fuentes = ["hidro", "geotermica_otras", "eolica", "bio", "solar", "fosil"]
colores = ["#2E86AB", "#8E6C8A", "#6BAA75", "#C9A227", "#F2C14E", OSCURO]
fig, ax = plt.subplots()
ax.stackplot(cr.index, [cr[f] for f in fuentes], labels=fuentes, colors=colores, alpha=.9)
ax.set_title("Generación por fuente (GWh)"); ax.legend(loc="upper left", ncol=6, fontsize=8, frameon=False)
plt.show()

print("Generación fósil por año (GWh):")
print(cr["fosil"].groupby(cr.index.year).sum().round(0))

**El año 2024.** Con El Niño llovió menos, los embalses bajaron y el país tuvo que encender plantas térmicas (la franja oscura): hubo racionamientos en mayo de 2024. Es un recordatorio de que los datos reales traen eventos que ningún modelo "sabe" de antemano.

Ahora miremos la estacionalidad: ¿qué meses consumen más?

In [ ]:
por_mes = cr.assign(mes_del_anio=cr.index.month).boxplot(column="demanda", by="mes_del_anio",
                                                         grid=False, figsize=(10, 4))
plt.suptitle(""); plt.title("Demanda por mes del año (GWh)"); plt.xlabel("mes"); plt.show()

prom = cr["demanda"].groupby(cr.index.month).mean()
print("Mes con mayor demanda promedio:", prom.idxmax(), "| menor:", prom.idxmin())

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
desc = seasonal_decompose(cr["demanda"], model="additive", period=12)
fig = desc.plot(); fig.set_size_inches(10, 7); plt.show()

print("Efecto estacional por mes (GWh sobre la tendencia):")
print(desc.seasonal.iloc[:12].round(1).set_axis(range(1, 13)))

**Lectura.** Marzo es el mes más alto: es época seca y calurosa y tiene 31 días. Febrero es el más bajo, sobre todo porque tiene menos días. La tendencia sube de forma casi lineal. En la prueba les pedimos exactamente esto: graficar, describir con palabras y descomponer.

## 3. Ecuaciones diferenciales: ¿a qué ritmo crece la demanda?

El modelo más simple de crecimiento es la EDO exponencial (Malthus, semana 2):

$$\frac{dD}{dt} = r\,D \quad\Rightarrow\quad D(t) = D_0\, e^{r t}$$

Tomando logaritmo, $\ln D = \ln D_0 + r\,t$: **una recta**. Así que estimamos $r$ con una regresión de $\ln D$ contra el año.

In [ ]:
from scipy.integrate import solve_ivp

completos = anual[(anual.index >= 2019) & (anual.index <= 2025)]
t = (completos.index - 2019).values
r, lnD0 = np.polyfit(t, np.log(completos.values), 1)
print(f"Tasa de crecimiento r = {r:.4f} por año  (≈ {100*(np.exp(r)-1):.1f} % anual)")

# Resolver la EDO numéricamente y comparar con la solución exacta
sol = solve_ivp(lambda t, D: r * D, [0, 11], [np.exp(lnD0)], t_eval=np.arange(0, 12))
exacta = np.exp(lnD0) * np.exp(r * sol.t)
print("Diferencia máxima numérica vs exacta:", float(np.max(np.abs(sol.y[0] - exacta))).__round__(4), "GWh")

fig, ax = plt.subplots()
ax.plot(completos.index, completos.values, "o", color=OSCURO, label="observado")
ax.plot(2019 + sol.t, sol.y[0], color=NARANJA, label="EDO dD/dt = rD")
ax.set_title("Demanda anual y modelo exponencial"); ax.set_ylabel("GWh"); ax.legend(); ax.grid(alpha=.3)
plt.show()
print(f"Proyección 2030 con la EDO: {np.exp(lnD0) * np.exp(r * 11):.0f} GWh")

**Para discutir.** Con siete puntos, la EDO da una tasa razonable, pero supone que el ritmo nunca cambia. Si el país electrifica el transporte, $r$ podría subir; si la economía se frena, bajar. Un modelo logístico ($dD/dt = rD(1 - D/K)$) agregaría un techo $K$, pero con estos datos no hay evidencia de saturación: no tiene sentido estimarlo (parsimonia, tema 1).

## 4. Cadenas de Markov: ¿cuándo se encienden las térmicas?

Definimos dos estados para cada mes:

- **R (renovable):** la generación fósil fue menor de 20 GWh.
- **T (térmica):** la generación fósil fue de 20 GWh o más.

Contamos cuántas veces se pasa de un estado al siguiente y dividimos por el total de cada fila: esa es la **matriz de transición estimada**.

In [ ]:
estado = np.where(cr["fosil"] >= 20, "T", "R")
print("Meses en T:", (estado == "T").sum(), "de", len(estado))

conteos = pd.crosstab(pd.Series(estado[:-1], name="hoy"), pd.Series(estado[1:], name="mañana"))
P = conteos.div(conteos.sum(axis=1), axis=0)
print("\nConteos de transiciones:\n", conteos)
print("\nMatriz de transición estimada:\n", P.round(3))

# Distribución estacionaria: resolver pi P = pi con sum(pi) = 1
A = np.vstack([P.values.T - np.eye(2), np.ones(2)])
b = np.array([0, 0, 1])
pi = np.linalg.lstsq(A, b, rcond=None)[0]
print("\nDistribución estacionaria (R, T):", pi.round(3))
print("Proporción observada     (R, T):", np.round([(estado == "R").mean(), (estado == "T").mean()], 3))

**Lectura.** Las filas suman 1. La fila T dice que, si un mes necesitó térmicas, es bastante probable que el siguiente también: los periodos secos **duran varios meses** (memoria). La distribución estacionaria dice qué fracción de los meses, a largo plazo, el país necesitaría térmicas si el clima se comportara como en 2019 a 2026. Ojo: la propiedad de Markov es un supuesto; un año con El Niño rompe la "memoria de un paso".

## 5. Regresión: tendencia y mes del año

Ajustamos $\text{demanda} = \beta_0 + \beta_1 t + \sum_{m=2}^{12} \gamma_m \,\mathbb{1}[\text{mes} = m] + \varepsilon$: una tendencia lineal más un efecto para cada mes (con enero como referencia).

Usamos la **misma partición** que en la sección 6: entrenamos hasta junio de 2025 y probamos de julio de 2025 a junio de 2026. En series de tiempo **no se baraja**.

In [ ]:
X = pd.get_dummies(cr.index.month, prefix="mes", drop_first=True, dtype=float)
X.index = cr.index
X.insert(0, "t", np.arange(len(cr)))
y = cr["demanda"]

corte = "2025-06-01"
X_tr, X_te = X[:corte], X[corte:].iloc[1:]
y_tr, y_te = y[:corte], y[corte:].iloc[1:]
print("Entrenamiento:", len(y_tr), "meses | Prueba:", len(y_te), "meses")

import statsmodels.api as sm
ols = sm.OLS(y_tr, sm.add_constant(X_tr)).fit()
print(f"\nPendiente: {ols.params['t']:.2f} GWh por mes  |  R² = {ols.rsquared:.3f}")
print("Efecto de marzo frente a enero:", round(ols.params["mes_3"], 1), "GWh")
print("Efecto de febrero frente a enero:", round(ols.params["mes_2"], 1), "GWh")

In [ ]:
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def rmse(a, b): return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))

cv = TimeSeriesSplit(n_splits=5)   # validación cruzada que respeta el orden del tiempo
modelos_reg = {
    "OLS": make_pipeline(StandardScaler(), LinearRegression()),
    "Ridge": make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 50), cv=cv)),
    "Lasso": make_pipeline(StandardScaler(), LassoCV(cv=cv, random_state=2026)),
}
for nombre, mod in modelos_reg.items():
    mod.fit(X_tr, y_tr)
    print(f"{nombre:6s} RMSE en prueba = {rmse(y_te, mod.predict(X_te)):.1f} GWh")
lasso = modelos_reg["Lasso"][-1]
print("\nLasso: alpha =", round(lasso.alpha_, 3), "| coeficientes en cero:", int(np.sum(lasso.coef_ == 0)), "de", len(lasso.coef_))

**Lectura.** La regresión con mes del año ya es un modelo de serie de tiempo: tendencia más estacionalidad, como Prophet. Aquí Ridge y Lasso no mejoran (incluso empeoran un poco): con 12 variables poco correlacionadas no hay sobreajuste que corregir, y encoger la pendiente hace que la tendencia se quede corta en el futuro. La regularización ayuda cuando hay **muchas** variables o variables **parecidas** entre sí. `TimeSeriesSplit` es la versión honesta de la validación cruzada para datos en el tiempo.

## 6. Series de tiempo: clásicos contra modernos

Este es el corazón de la prueba. El plan:

1. Misma partición: entrenar hasta junio de 2025, probar los 12 meses siguientes.
2. Una **línea base**: el ingenuo estacional (cada mes repite el mismo mes del año anterior).
3. Dos clásicos: **Holt-Winters** (suavizamiento exponencial) y **SARIMA**.
4. Dos modernos: **Prophet** y **Chronos-2** (zero-shot).
5. Las mismas métricas para todos: **RMSE**, **MAE** y **MASE**.

In [ ]:
def mae(a, b): return float(np.mean(np.abs(np.asarray(a) - np.asarray(b))))
def mase(real, pron, entrenamiento, m=12):
    escala = np.mean(np.abs(entrenamiento[m:] - entrenamiento[:-m]))
    return float(np.mean(np.abs(np.asarray(real) - np.asarray(pron))) / escala)

serie = cr["demanda"]
train, test = serie[:"2025-06"], serie["2025-07":]
h = len(test)
pron = {}
pron["Ingenuo estacional"] = train.values[-12:]
print("Train:", train.index.min().date(), "a", train.index.max().date(), "| Test:", h, "meses")

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX

hw = ExponentialSmoothing(train, trend="add", seasonal="add", seasonal_periods=12).fit()
pron["Holt-Winters"] = hw.forecast(h).values

sarima = SARIMAX(train, order=(1, 0, 0), seasonal_order=(0, 1, 1, 12), trend="c").fit(disp=False)
pron["SARIMA"] = sarima.forecast(h).values
print(sarima.summary().tables[1])

**¿Por qué SARIMA(1,0,0)(0,1,1)₁₂ con constante?** Una diferencia estacional ($D = 1$) quita el patrón anual; la constante hace el papel de la deriva (lo que crece la demanda de un año al siguiente, como en la semana 6); el AR(1) recoge la memoria de un mes al siguiente. Es un modelo pequeño y razonable, no el "mejor posible": en la prueba pueden justificar otro.

In [ ]:
from prophet import Prophet
m_prophet = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
m_prophet.fit(train.reset_index().rename(columns={"mes": "ds", "demanda": "y"}))
fc_prophet = m_prophet.predict(m_prophet.make_future_dataframe(periods=h, freq="MS"))
pron["Prophet"] = fc_prophet["yhat"].values[-h:]

from chronos import Chronos2Pipeline
chronos = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map="cpu")
contexto = train.reset_index().rename(columns={"mes": "timestamp", "demanda": "target"}).assign(id="CR")
fc_chronos = chronos.predict_df(contexto, prediction_length=h, quantile_levels=[0.1, 0.5, 0.9],
                                id_column="id", timestamp_column="timestamp", target="target")
pron["Chronos-2"] = fc_chronos["0.5"].values
print("Cinco pronósticos listos:", list(pron))

In [ ]:
tabla = pd.DataFrame({
    nombre: {"RMSE": rmse(test, f), "MAE": mae(test, f), "MASE": mase(test, f, train.values)}
    for nombre, f in pron.items()}).T.sort_values("MASE")
print(tabla.round(3))

fig, ax = plt.subplots()
ax.plot(serie["2023-01":].index, serie["2023-01":], color=OSCURO, lw=2, label="real")
for (nombre, f), c in zip(pron.items(), [GRIS, "#2E86AB", NARANJA, "#C0392B", "#6BAA75"]):
    ax.plot(test.index, f, color=c, ls="--" if nombre.startswith("Ingenuo") else "-", label=nombre)
ax.axvline(test.index[0], color=GRIS, ls=":"); ax.legend(ncol=3, fontsize=9, frameon=False)
ax.set_title("Pronósticos de julio 2025 a junio 2026 contra lo real (GWh)"); plt.show()

**Lectura.** Todos le ganan al ingenuo estacional, y el SARIMA queda primero. Pero esto es **un solo año de prueba**. Antes de elegir, repetimos el ejercicio en varias ventanas (ventana expansiva, semana 6): cada vez se entrena con todo lo anterior y se pronostican 12 meses.

In [ ]:
warnings.filterwarnings("ignore")   # statsmodels avisa sobre valores iniciales; no afecta el resultado

def pronosticar(tr, h=12):
    out = {"Ingenuo estacional": tr.values[-12:]}
    out["Holt-Winters"] = ExponentialSmoothing(tr, trend="add", seasonal="add",
                                               seasonal_periods=12).fit().forecast(h).values
    out["SARIMA"] = SARIMAX(tr, order=(1, 0, 0), seasonal_order=(0, 1, 1, 12),
                            trend="c").fit(disp=False).forecast(h).values
    ctx = tr.reset_index().rename(columns={"mes": "timestamp", "demanda": "target"}).assign(id="CR")
    out["Chronos-2"] = chronos.predict_df(ctx, prediction_length=h, quantile_levels=[0.5], id_column="id",
                                          timestamp_column="timestamp", target="target")["0.5"].values
    return out

filas = []
for inicio in ["2023-07", "2024-07", "2025-07"]:
    tr = serie[serie.index < inicio]
    te = serie[inicio:].iloc[:12]
    for nombre, f in pronosticar(tr).items():
        filas.append({"prueba desde": inicio, "modelo": nombre, "MASE": mase(te, f, tr.values)})
cv_ts = pd.DataFrame(filas).pivot(index="modelo", columns="prueba desde", values="MASE")
cv_ts["promedio"] = cv_ts.mean(axis=1)
print(cv_ts.sort_values("promedio").round(3))

**Lectura: la elección cambia.** Con un solo año de prueba habríamos elegido el SARIMA. Pero en la ventana que empieza en julio de 2023 (la que contiene la sequía de 2024 y su salto de demanda) el SARIMA empata con el ingenuo (MASE ≈ 1.00), mientras Holt-Winters se mantiene alrededor de 0.5 en las tres ventanas. Prophet se dejó fuera de este bucle solo por tiempo de ejecución; en la prueba pueden incluirlo.

**Regla para elegir:** el modelo más simple que le gane al ingenuo de forma consistente (MASE promedio menor que 1 y ninguna ventana desastrosa). Aquí gana **Holt-Winters**: mejor promedio y el más estable. Chronos-2, sin entrenar, queda segundo.

## 7. Monte Carlo: ¿qué tan probable es superar cierto consumo?

Un pronóstico puntual no basta para decidir. Reentrenamos Holt-Winters con **todos** los datos y simulamos 5000 futuros posibles de 12 meses: cada futuro suma al pronóstico errores aleatorios del tamaño de los que el modelo cometió en el pasado. Con esas simulaciones respondemos preguntas de probabilidad, como en la semana 4.

In [ ]:
final = ExponentialSmoothing(serie, trend="add", seasonal="add", seasonal_periods=12).fit()
sims = final.simulate(nsimulations=12, repetitions=5000, anchor="end", error="add", random_state=2026)
sims = np.asarray(sims)                     # 12 meses x 5000 futuros
total = sims.sum(axis=0)                    # demanda total de cada futuro simulado

umbral = 14000
print(f"Referencia: los últimos 12 meses observados sumaron {serie[-12:].sum():.0f} GWh")
print(f"Demanda total jul 2026 - jun 2027: media {total.mean():.0f} GWh")
print(f"Intervalo del 90 %: {np.percentile(total, 5):.0f} a {np.percentile(total, 95):.0f} GWh")
print(f"P(total > {umbral} GWh) = {np.mean(total > umbral):.3f}")

plt.hist(total, bins=50, color=NARANJA, edgecolor="white")
plt.axvline(umbral, color=OSCURO, ls="--", label=f"{umbral} GWh"); plt.legend()
plt.title("5000 futuros simulados: demanda total del próximo año"); plt.xlabel("GWh"); plt.show()

**Ojo crítico.** Comparen la media simulada con lo observado en los últimos 12 meses: el modelo espera un crecimiento cercano al 7 %, por encima del ritmo de la EDO (2.7 % anual). La tendencia aditiva de Holt-Winters extrapola el empuje reciente; con `damped_trend=True` se amortigua. Un buen modelador revisa si el pronóstico tiene sentido, no solo su MASE.

**Monte Carlo o MCMC.** Aquí simulamos futuros de un modelo ya ajustado: eso es **Monte Carlo**. MCMC (Metropolis, Gibbs, PyMC) también simula, pero para otra cosa: obtener muestras de la **distribución posterior de los parámetros** cuando no se puede calcular directamente.

## 8. Pronóstico final

Con el modelo elegido y **todos** los datos, pronosticamos julio de 2026 a junio de 2027. La banda del 80 % sale de las mismas simulaciones de Monte Carlo (percentiles 10 y 90 de cada mes).

In [ ]:
media = final.forecast(12)
p10, p90 = np.percentile(sims, 10, axis=1), np.percentile(sims, 90, axis=1)

fig, ax = plt.subplots()
ax.plot(serie.index, serie, color=OSCURO, label="observado")
ax.plot(media.index, media, color=NARANJA, marker="o", ms=3, label="pronóstico Holt-Winters")
ax.fill_between(media.index, p10, p90, color=NARANJA, alpha=.2, label="banda 80 % (Monte Carlo)")
ax.set_title("Demanda eléctrica: pronóstico a 12 meses (GWh)"); ax.legend(); ax.grid(alpha=.3)
plt.show()

salida = pd.DataFrame({"mes": media.index.strftime("%Y-%m"), "pronostico_gwh": media.round(1).values,
                       "p10": p10.round(1), "p90": p90.round(1)})
print(salida.to_string(index=False))

## 9. Checklist para la prueba

**Errores frecuentes**

- Barajar los datos o usar K-fold normal en una serie: se filtra el futuro.
- Comparar modelos en conjuntos de prueba distintos o con métricas distintas.
- Elegir con un solo año de prueba sin mirar si el resultado se repite (hoy habría elegido el modelo equivocado).
- Decir "el modelo es bueno" sin compararlo con el ingenuo estacional.
- Interpretar un MASE mayor que 1 como "malo" sin mirar al ingenuo en la prueba.
- Celdas que solo corren en cierto orden o variables que no se definieron antes.

**Formulario mínimo**

| Concepto | Fórmula |
|---|---|
| Mínimos cuadrados | $\hat\beta_1 = \dfrac{\sum (x_i-\bar x)(y_i-\bar y)}{\sum (x_i-\bar x)^2}$, $\ \hat\beta_0 = \bar y - \hat\beta_1 \bar x$ |
| MAE | $\frac{1}{n}\sum \lvert y_t - \hat y_t \rvert$ |
| RMSE | $\sqrt{\frac{1}{n}\sum (y_t - \hat y_t)^2}$ |
| MASE | $\dfrac{\text{MAE del modelo}}{\frac{1}{T-m}\sum_{t=m+1}^{T} \lvert y_t - y_{t-m}\rvert}$ |
| EDO exponencial | $dD/dt = rD \Rightarrow D(t) = D_0 e^{rt}$ |
| Estacionaria | $\pi P = \pi$, $\ \sum_i \pi_i = 1$ |

**La prueba en una frase:** descargar una serie real, explorarla, comparar al menos un clásico y un moderno contra el ingenuo con las mismas métricas, validar en más de una ventana, elegir con un criterio explícito y pronosticar.